# Create Smoking Research Foundation Awards (公益財団法人 喫煙科学研究財団)

Creates Smoking Research Foundation (Tokyo, Japan) research-grant awards from the
foundation's own "Former studies" lists (https://www.srf.or.jp/en/studies.html, rendered
by the POST form at https://www.srf.or.jp/app_list2/). Each row is one funded study:
title, period (start-end fiscal year) and, for 73%, a link to its results-summary PDF
whose file name carries the study number. **1,760 completed studies, 1986-2025:
General Research (11 field lists) 1,361, Project Research 280, Early-Career Scientists
Research 119** (local run 2026-10-01, oxjob #1451). The Heated Tobacco Product list is a
cross-listing of studies already in the field lists (tagged, not duplicated).

**What SRF does not publish:** no per-study PI, institution or amount (the foundation
pays standard rates, e.g. 2M JPY/year for General Research), and the summary PDFs are
anonymous. So `lead_investigator`, `amount` and `currency` are NULL; the §6.7 amount
check is waived (funder does not publish per-grant amounts). Dates are fiscal years
only: `start_year`/`end_year` from the period, `start_date`/`end_date` NULL.

**Not ingested:** the "Current studies" list (FY2026, 179 ongoing studies) has neither a
period nor a study number; those studies move into the Former lists, with their number,
when they finish, so a later refresh picks them up without churning synthetic ids.
No non-research money is involved; nothing filtered.

**Prerequisites:** run `scripts/local/smoking_research_foundation_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/smoking_research_foundation/smoking_research_foundation_projects.parquet`
(§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** the study number from the report PDF name: `FP` + 8
digits (1,007 studies, mostly to the FY2018 reports) or `YYYY{G|Y|T}NNN` (281 studies
from ~2017 starts; G general, Y early-career, T project research). Citing works use both
forms (`FP01606073`, `2022G002`); 35 existing acknowledgement stub ids collapse onto this
ingest. Studies with no PDF (mostly 1986-1995) and the 7 Project Research sub-studies that
share one group report (FP01112200) ship a stable synthetic key
`SRF-{period}-{md5(lower(english title))[:8]}`. Many 2014-2018 studies are cited with the
application-style number (`2016Y008`) while the site only shows their FP report number,
so those stubs cannot collapse.

`display_name` = English title (the Japanese title, where the Japanese list carries the
same study number, is kept in the raw table as `title_ja`). `landing_page_url` = the
study's English results-summary PDF when there is one, else the studies page.
`funder_scheme` = field list / Project Research theme / Early-Career.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320322555`
- display_name: Smoking Research Foundation
- ror_id / doi: from `openalex.funders.funders` (ror 04q4hbm31, doi 10.13039/501100004330)

**Priority:** `541` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.smoking_research_foundation_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/smoking_research_foundation/smoking_research_foundation_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.smoking_research_foundation_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.smoking_research_foundation_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.smoking_research_foundation_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320322555 is a Crossref-registered F4320* funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320322555;

## Step 2: Create Smoking Research Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.smoking_research_foundation_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320322555  -- Smoking Research Foundation
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title_en), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,      -- SRF publishes no per-study amount
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    g.funder_scheme as funder_scheme,
    'srf_former_studies' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.start_year AS INT) as start_year,
    TRY_CAST(g.end_year AS INT) as end_year,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as lead_investigator,   -- SRF publishes no PI
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    COALESCE(NULLIF(TRIM(g.report_pdf_en), ''), g.landing_page_url) as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.smoking_research_foundation_raw g
CROSS JOIN src_funder f
WHERE NULLIF(TRIM(g.funder_award_id), '') IS NOT NULL;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'srf_former_studies' AND priority = 541;

-- Insert into openalex_awards_raw with priority.
-- Priority 541: direct-from-funder ingest (higher wins under the 2026-06-20 DESC
-- dedup, oxjob #500), so it outranks the priority-0 acknowledgement shells that
-- share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    541 as priority
FROM openalex.awards.smoking_research_foundation_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids,
       COUNT(DISTINCT LOWER(funder_award_id)) as distinct_award_ids
FROM openalex.awards.smoking_research_foundation_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, lead_investigator.affiliation.country
FROM openalex.awards.smoking_research_foundation_awards LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage
SELECT
    COUNT(*) as total,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    collect_set(currency) as currencies,
    MIN(amount) as min_amt, MAX(amount) as max_amt, ROUND(AVG(amount), 0) as avg_amt,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_start_date,
    ROUND(COUNT(start_year) * 100.0 / COUNT(*), 1) as pct_start_year,
    ROUND(COUNT(lead_investigator.family_name) * 100.0 / COUNT(*), 1) as pct_pi,
    ROUND(COUNT(lead_investigator.affiliation.name) * 100.0 / COUNT(*), 1) as pct_institution
FROM openalex.awards.smoking_research_foundation_awards;

In [ ]:
%sql
SELECT funder_scheme, currency, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_amount
FROM openalex.awards.smoking_research_foundation_awards
GROUP BY funder_scheme, currency ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.smoking_research_foundation_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.smoking_research_foundation_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.smoking_research_foundation_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.5 funder consistency
SELECT funder.display_name, funder_id, COUNT(*)
FROM openalex.awards.smoking_research_foundation_awards
GROUP BY funder.display_name, funder_id ORDER BY 3 DESC;

In [ ]:
%sql
-- Award-id shape (2.1.1): FP + 8 digits, YYYY{G|Y|T}NNN, or the documented synthetic key.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^FP[0-9]{8}$' THEN 1 ELSE 0 END) AS fp_number,
    SUM(CASE WHEN funder_award_id RLIKE '^[0-9]{4}[GYT][0-9]{3}$' THEN 1 ELSE 0 END) AS new_number,
    SUM(CASE WHEN funder_award_id RLIKE '^SRF-[0-9]{4}-[0-9]{4}-[0-9a-f]{8}$' THEN 1 ELSE 0 END) AS synthetic_key,
    SUM(CASE WHEN NOT (funder_award_id RLIKE '^FP[0-9]{8}$' OR funder_award_id RLIKE '^[0-9]{4}[GYT][0-9]{3}$'
                       OR funder_award_id RLIKE '^SRF-[0-9]{4}-[0-9]{4}-[0-9a-f]{8}$') THEN 1 ELSE 0 END) AS other_shape -- must be 0
FROM openalex.awards.smoking_research_foundation_awards;

In [ ]:
%sql
-- Existing acknowledgement stubs that collapse onto this ingest (expect ~35 distinct ids).
SELECT c.provenance, COUNT(*) AS stub_rows, COUNT(s.id) AS collapsed
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.smoking_research_foundation_awards s ON s.id = c.id
WHERE c.funder_id = 4320322555 AND c.priority = 0
GROUP BY c.provenance;

In [ ]:
%sql
-- Section 6.8: rows reached the shared raw table at the assigned priority.
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'srf_former_studies'
GROUP BY provenance, priority;